# Books to Scrape — Advanced Concurrent Web Scraper

This notebook crawls **all 50 categories**, **all paginated listing pages**, and **every book's detail page** on [books.toscrape.com](https://books.toscrape.com/) to produce a clean, validated dataset of all **1,000 books**.

### Problems Solved vs. Original Scraper
1. **Complete Pagination**: Follows `li.next a` across all category pages (1,000 books instead of 517).
2. **UTF-8 Encoding Fix**: Decodes raw UTF-8 response bytes directly, eliminating mojibake (`Â£`, `â€™`, `â€œ`, `Ã©`, etc.).
3. **URL Safety**: Uses `urllib.parse.urljoin` for all category, pagination, product, and cover image URLs.
4. **HTTP Resilience & Concurrency**: Uses thread-local `requests.Session` instances with connection pooling, timeouts, exponential-backoff retries (`urllib3.Retry`), and `ThreadPoolExecutor`.
5. **Rich Typed Schema & Validation**: Extracts `UPC`, `Title`, `Category`, `Rating` (1–5), `Price_Excl_Tax`, `Price_Incl_Tax`, `Tax`, `In_Stock`, `Stock_Quantity`, `Num_Reviews`, `Description`, `Product_URL`, and `Image_URL`, then validates invariants and writes `books.csv` + `scraping_summary.json`.

In [1]:
import csv
import json
import logging
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://books.toscrape.com/"
REQUEST_TIMEOUT = 15
MAX_WORKERS = 12
USER_AGENT = "NTI-BookScraper/2.0 (Educational Portfolio Pipeline)"

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

MOJIBAKE_TOKENS = ("Â£", "â€™", "â€œ", "â€", "â€”", "Ã©", "Ã¡", "Ã±")

# Resolve output directory whether running from repo root or 'Scraping Books/'
CWD = Path.cwd()
OUTPUT_DIR = CWD / "Scraping Books" if (CWD / "Scraping Books").is_dir() else CWD
BOOKS_CSV_PATH = OUTPUT_DIR / "books.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "scraping_summary.json"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("books_scraper")
logger.info("Output directory: %s", OUTPUT_DIR.resolve())

19:43:37 | INFO | Output directory: C:\Users\L0Q\Desktop\scraping\Scraping Books


In [2]:
_thread_local = threading.local()


def get_session() -> requests.Session:
    """Return a thread-local Session configured with retries and connection pooling."""
    session = getattr(_thread_local, "session", None)
    if session is not None:
        return session

    retry_strategy = Retry(
        total=4,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=MAX_WORKERS,
        pool_maxsize=MAX_WORKERS,
    )
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    _thread_local.session = session
    return session


def fetch_soup(url: str) -> BeautifulSoup:
    """Fetch a URL and parse raw UTF-8 response bytes with BeautifulSoup."""
    session = get_session()
    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    # Parse raw bytes with explicit UTF-8 so requests' ISO-8859-1 fallback never causes mojibake
    return BeautifulSoup(response.content, "lxml", from_encoding="utf-8")


def parse_money(raw: str) -> Optional[float]:
    """Extract a numeric GBP price from strings like '£51.77'."""
    match = re.search(r"\d+(?:\.\d+)?", raw.replace(",", ""))
    return round(float(match.group()), 2) if match else None


def parse_rating(tag) -> Optional[int]:
    """Map BeautifulSoup star-rating CSS class to integer 1..5."""
    if tag is None:
        return None
    for cls in tag.get("class", []):
        if cls in RATING_MAP:
            return RATING_MAP[cls]
    return None


def parse_availability(raw: str) -> tuple[bool, int]:
    """Return (in_stock_bool, stock_quantity) from text like 'In stock (22 available)'."""
    normalized = " ".join(raw.split())
    in_stock = "in stock" in normalized.lower()
    qty_match = re.search(r"(\d+)\s+available", normalized, flags=re.IGNORECASE)
    quantity = int(qty_match.group(1)) if qty_match else (1 if in_stock else 0)
    return in_stock, quantity

In [3]:
@dataclass
class ListingStub:
    category: str
    listing_title: str
    listing_rating: Optional[int]
    product_url: str


@dataclass
class BookRecord:
    UPC: str
    Title: str
    Category: str
    Rating: Optional[int]
    Price_Excl_Tax: Optional[float]
    Price_Incl_Tax: Optional[float]
    Tax: Optional[float]
    In_Stock: bool
    Stock_Quantity: int
    Num_Reviews: int
    Description: str
    Product_URL: str
    Image_URL: str


def discover_categories(base_url: str) -> list[tuple[str, str]]:
    """Discover all leaf book categories and their absolute URLs from the homepage."""
    soup = fetch_soup(base_url)
    category_links = soup.select("div.side_categories ul li ul li a")
    categories = [
        (" ".join(link.text.split()), urljoin(base_url, link["href"]))
        for link in category_links
    ]
    return categories


def crawl_category_listings(category_name: str, start_url: str) -> list[ListingStub]:
    """Follow all paginated pages for one category and collect every book URL."""
    stubs: list[ListingStub] = []
    current_url: Optional[str] = start_url

    while current_url:
        soup = fetch_soup(current_url)
        for card in soup.select("article.product_pod"):
            link_tag = card.select_one("h3 a")
            if link_tag is None:
                continue
            title = link_tag.get("title") or " ".join(link_tag.text.split())
            product_url = urljoin(current_url, link_tag["href"])
            rating = parse_rating(card.select_one("p.star-rating"))
            stubs.append(ListingStub(category_name, title.strip(), rating, product_url))

        next_link = soup.select_one("li.next a")
        current_url = urljoin(current_url, next_link["href"]) if next_link else None

    return stubs


def scrape_book_detail(stub: ListingStub) -> BookRecord:
    """Fetch a book's detail page and parse full product attributes."""
    soup = fetch_soup(stub.product_url)

    title_tag = soup.select_one("div.product_main h1")
    title = " ".join(title_tag.text.split()) if title_tag else stub.listing_title

    rating = parse_rating(soup.select_one("div.product_main p.star-rating")) or stub.listing_rating

    desc_tag = soup.select_one("#product_description + p")
    description = " ".join(desc_tag.text.split()) if desc_tag else ""

    img_tag = soup.select_one("div.item.active img") or soup.select_one("div#product_gallery img")
    image_url = urljoin(stub.product_url, img_tag["src"]) if img_tag and img_tag.get("src") else ""

    info_table: dict[str, str] = {}
    for row in soup.select("table.table.table-striped tr"):
        th = row.select_one("th")
        td = row.select_one("td")
        if th and td:
            info_table[" ".join(th.text.split())] = " ".join(td.text.split())

    in_stock, stock_qty = parse_availability(info_table.get("Availability", ""))
    reviews_raw = info_table.get("Number of reviews", "0")
    num_reviews = int(reviews_raw) if reviews_raw.isdigit() else 0

    return BookRecord(
        UPC=info_table.get("UPC", ""),
        Title=title,
        Category=stub.category,
        Rating=rating,
        Price_Excl_Tax=parse_money(info_table.get("Price (excl. tax)", "")),
        Price_Incl_Tax=parse_money(info_table.get("Price (incl. tax)", "")),
        Tax=parse_money(info_table.get("Tax", "")),
        In_Stock=in_stock,
        Stock_Quantity=stock_qty,
        Num_Reviews=num_reviews,
        Description=description,
        Product_URL=stub.product_url,
        Image_URL=image_url,
    )

In [4]:
def run_scraper() -> tuple[pd.DataFrame, list[dict]]:
    """Execute category discovery, paginated listing crawl, and concurrent detail scraping."""
    start_time = time.perf_counter()
    categories = discover_categories(BASE_URL)
    logger.info("Discovered %d categories", len(categories))

    all_stubs: list[ListingStub] = []
    errors: list[dict] = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        cat_futures = {
            executor.submit(crawl_category_listings, name, url): (name, url)
            for name, url in categories
        }
        for future in as_completed(cat_futures):
            cat_name, cat_url = cat_futures[future]
            try:
                all_stubs.extend(future.result())
            except Exception as exc:
                errors.append({"stage": "category_crawl", "category": cat_name, "url": cat_url, "error": str(exc)})
                logger.error("Category crawl failed for %s: %s", cat_name, exc)

    logger.info("Collected %d listing stubs across %d categories", len(all_stubs), len(categories))

    records: list[dict] = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        detail_futures = {
            executor.submit(scrape_book_detail, stub): stub
            for stub in all_stubs
        }
        for future in as_completed(detail_futures):
            stub = detail_futures[future]
            try:
                records.append(asdict(future.result()))
            except Exception as exc:
                errors.append({"stage": "detail_scrape", "category": stub.category, "url": stub.product_url, "error": str(exc)})
                logger.error("Detail scrape failed for %s: %s", stub.product_url, exc)

    elapsed = round(time.perf_counter() - start_time, 2)
    df = pd.DataFrame(records)
    logger.info("Scraped %d books in %.2fs (%d errors)", len(df), elapsed, len(errors))
    return df, errors


books_df, scrape_errors = run_scraper()
print(f"Raw scraped rows: {len(books_df)} | Errors: {len(scrape_errors)}")

19:43:39 | INFO | Discovered 50 categories


19:43:43 | INFO | Collected 1000 listing stubs across 50 categories


19:44:05 | INFO | Scraped 1000 books in 27.62s (0 errors)


Raw scraped rows: 1000 | Errors: 0


In [5]:
def validate_and_export(df: pd.DataFrame, errors: list[dict]) -> dict:
    """Validate scraped records, deduplicate by UPC, export CSV & JSON summary."""
    if df.empty:
        raise RuntimeError("Scraper returned zero records; aborting export.")

    df = df.copy()
    raw_count = len(df)

    # Deterministic sort & UPC deduplication
    df = df.sort_values(["Category", "Title", "UPC"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["UPC"], keep="first").reset_index(drop=True)

    # Check for mojibake across string fields
    text_cols = ["Title", "Category", "Description"]
    mojibake_hits = 0
    for col in text_cols:
        series = df[col].fillna("").astype(str)
        for token in MOJIBAKE_TOKENS:
            mojibake_hits += int(series.str.contains(token, regex=False).sum())

    # Validate invariants
    missing_upc = int((df["UPC"].str.len() == 0).sum())
    invalid_ratings = int((~df["Rating"].isin([1, 2, 3, 4, 5])).sum())
    invalid_prices = int(((df["Price_Incl_Tax"] < df["Price_Excl_Tax"]) | (df["Price_Excl_Tax"] <= 0)).sum())
    duplicate_titles_distinct_upc = int(df["Title"].duplicated(keep=False).sum())

    # Export clean UTF-8 CSV
    df.to_csv(BOOKS_CSV_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

    summary = {
        "scraped_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_url": BASE_URL,
        "raw_rows_scraped": raw_count,
        "unique_books_exported": len(df),
        "categories_count": int(df["Category"].nunique()),
        "in_stock_count": int(df["In_Stock"].sum()),
        "total_stock_units": int(df["Stock_Quantity"].sum()),
        "books_without_description": int((df["Description"].str.len() == 0).sum()),
        "duplicate_titles_with_distinct_upc": duplicate_titles_distinct_upc,
        "price_incl_tax_min_gbp": float(df["Price_Incl_Tax"].min()),
        "price_incl_tax_median_gbp": round(float(df["Price_Incl_Tax"].median()), 2),
        "price_incl_tax_mean_gbp": round(float(df["Price_Incl_Tax"].mean()), 2),
        "price_incl_tax_max_gbp": float(df["Price_Incl_Tax"].max()),
        "rating_distribution": {str(k): int(v) for k, v in df["Rating"].value_counts().sort_index().items()},
        "validation_checks": {
            "mojibake_hits": mojibake_hits,
            "missing_upc": missing_upc,
            "invalid_ratings": invalid_ratings,
            "invalid_prices": invalid_prices,
            "scrape_error_count": len(errors),
        },
        "errors": errors,
    }

    SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return summary


scrape_summary = validate_and_export(books_df, scrape_errors)
print(json.dumps(scrape_summary, indent=2, ensure_ascii=False))
books_df.head(5)[["UPC", "Category", "Title", "Rating", "Price_Incl_Tax", "Stock_Quantity"]]

{
  "scraped_at_utc": "2026-11-22T17:44:05+00:00",
  "source_url": "https://books.toscrape.com/",
  "raw_rows_scraped": 1000,
  "unique_books_exported": 1000,
  "categories_count": 50,
  "in_stock_count": 1000,
  "total_stock_units": 8585,
  "books_without_description": 2,
  "duplicate_titles_with_distinct_upc": 2,
  "price_incl_tax_min_gbp": 10.0,
  "price_incl_tax_median_gbp": 35.98,
  "price_incl_tax_mean_gbp": 35.07,
  "price_incl_tax_max_gbp": 59.99,
  "rating_distribution": {
    "1": 226,
    "2": 196,
    "3": 203,
    "4": 179,
    "5": 196
  },
  "validation_checks": {
    "mojibake_hits": 0,
    "missing_upc": 0,
    "invalid_ratings": 0,
    "invalid_prices": 0,
    "scrape_error_count": 0
  },
  "errors": []
}


,UPC,Category,Title,Rating,Price_Incl_Tax,Stock_Quantity
0,9e60929f521fa280,Travel,A Year in Provence (Provence #1),4,56.88,6
1,366a236aa1ea6f07,Travel,The Road to Little Dribbling: Adventures of an...,1,23.21,3
2,48736df57e7bec9f,Travel,The Great Railway Bazaar,1,30.54,6
3,ce60436f52c5ee68,Travel,Full Moon over Noah’s Ark: An Odyssey to Mount...,4,49.43,15
4,a22124811bfa8350,Travel,It's Only the Himalayas,2,45.17,19
